In [ ]:
"""
Colab 服务端执行脚本 (基于 Google Drive 文件状态机)。
增加强制云盘刷新机制，解决同步延迟与缓存问题。
发现 start 即转换为 running 并执行命令，结束后回写 success/fail。
"""

import os
import json
import time
import shlex
import subprocess

# ==========================================
# 配置区域 (Configuration)
# ==========================================

# Google Drive 挂载与联动配置 (Linux环境路径)
GDRIVE_MOUNT_POINT = "/content/drive"
DRIVE_BASE_DIR = "/content/drive/MyDrive/AiGenerations"
TASK_FILE_PATH = os.path.join(DRIVE_BASE_DIR, "task_file.txt")

# 轮询配置
POLL_INTERVAL_SECONDS = 3

# ==========================================
# 底层功能函数 (Core Functions)
# ==========================================

def mount_google_drive_and_init():
    """挂载 Google Drive 并确保基础目录存在"""
    print("[系统初始化] 准备挂载 Google Drive...")
    try:
        from google.colab import drive # pylint: disable=import-outside-toplevel
        drive.mount(GDRIVE_MOUNT_POINT)
        print(f"[系统初始化] [成功] 挂载至: {GDRIVE_MOUNT_POINT}")
        
        if not os.path.exists(DRIVE_BASE_DIR):
            os.makedirs(DRIVE_BASE_DIR)
            print(f"[系统初始化] [成功] 创建同步基础目录: {DRIVE_BASE_DIR}")
        return True
    except ImportError:
        print("[系统初始化] [失败] 未检测到 Colab 环境。")
        return False
    except Exception as exc: # pylint: disable=broad-exception-caught
        print(f"[系统初始化] [异常] 挂载崩溃: {exc}")
        return False

def force_drive_cache_refresh():
    """通过读取目录列表，强制 Colab 刷新 Google Drive 的底层缓存"""
    try:
        if os.path.exists(DRIVE_BASE_DIR):
            _ = os.listdir(DRIVE_BASE_DIR)
            return True
        print(f"[缓存刷新] [失败] 基础目录不存在: {DRIVE_BASE_DIR}")
        return False
    except Exception as exc: # pylint: disable=broad-exception-caught
        print(f"[缓存刷新] [异常] 强制刷新目录时发生异常: {exc}")
        return False

def read_task_file():
    """安全读取并解析 task_file.txt，增加详细异常抛出"""
    if not os.path.exists(TASK_FILE_PATH):
        # 文件尚未同步过来，保持静默，不污染日志
        return None
        
    try:
        with open(TASK_FILE_PATH, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            
        if not content:
            print("[文件读取] [警告] 读取到了空白文件，云盘可能正在写入中...")
            return None
            
        return json.loads(content)
    except json.JSONDecodeError as exc:
        print(f"[文件读取] [错误] JSON 解析失败。云盘可能只同步了一半的数据: {exc}")
        return None
    except Exception as exc: # pylint: disable=broad-exception-caught
        print(f"[文件读取] [异常] 读取文件时发生系统崩溃: {exc}")
        return None

def update_task_status(task_data, new_status, video_rel_path=None):
    """更新 task_file.txt 中的状态信息，并回写"""
    print(f"[状态更新] 准备将状态修改为: {new_status}")
    try:
        task_data["status"] = new_status
        if video_rel_path:
            task_data["video"] = video_rel_path
            
        with open(TASK_FILE_PATH, 'w', encoding='utf-8') as f:
            json.dump(task_data, f, ensure_ascii=False, indent=4)
            
        print("[状态更新] [成功] 状态回写云盘成功。")
        return True
    except Exception as exc: # pylint: disable=broad-exception-caught
        print(f"[状态更新] [异常] 回写状态失败: {exc}")
        return False

def execute_ffmpeg(command_str):
    """执行 FFmpeg 核心命令"""
    print(f"[引擎执行] 准备执行系统命令: {command_str}")
    try:
        cmd_list = shlex.split(command_str)
        process = subprocess.run(cmd_list, capture_output=True, text=True, check=False)
        
        if process.returncode != 0:
            print(f"[引擎执行] [失败] FFmpeg 报错退出，状态码: {process.returncode}")
            print(f"[引擎执行] [错误日志] {process.stderr}")
            return False
            
        print("[引擎执行] [成功] 视频生成处理完毕。")
        return True
    except Exception as exc: # pylint: disable=broad-exception-caught
        print(f"[引擎执行] [异常] 子进程执行发生崩溃: {exc}")
        return False

def process_pending_task():
    """主轮询逻辑：检查是否有 start 状态的任务并处理"""
    
    # 第一步：强制刷新缓存，打破 Colab 的文件盲区
    if not force_drive_cache_refresh():
        return False

    # 第二步：读取数据
    task_data = read_task_file()
    if not task_data:
        return True # 无文件或文件为空/残缺，继续挂机
        
    status = task_data.get("status")
    
    # 只要不是 start 状态，云端就不做任何处理
    if status != "start":
        return True 

    # 1. 发现新任务，将状态变更为 running
    print("\n---> [任务捕获] 成功发现新指令！启动处理流水线 <---")
    if not update_task_status(task_data, "running"):
        print("[流水线阻断] [严重错误] 无法将状态更新为 running，拒绝执行后续操作。")
        return False
        
    # 2. 解析相对路径并组装 Colab 的绝对路径
    relative_image_path = task_data.get("image")
    command_template = task_data.get("command")
    
    if not relative_image_path or not command_template:
        print("[任务执行] [错误] 任务文件格式残缺，缺少 image 或 command 核心字段。")
        update_task_status(task_data, "fail")
        return False
        
    # 拼接绝对路径
    abs_image_path = os.path.join(DRIVE_BASE_DIR, relative_image_path)
    
    # 视频输出路径同目录同名，改后缀
    task_dir = os.path.dirname(relative_image_path)
    task_id = os.path.basename(task_dir)
    relative_video_path = f"{task_dir}/{task_id}.mp4"
    abs_video_path = os.path.join(DRIVE_BASE_DIR, relative_video_path)
    
    # 3. 替换命令模板中的占位符
    final_command = command_template.replace("{INPUT_IMAGE}", abs_image_path)
    final_command = final_command.replace("{OUTPUT_VIDEO}", abs_video_path)
    
    # 4. 执行生成操作
    is_success = execute_ffmpeg(final_command)
    
    # 5. 更新最终状态
    if is_success:
        print("[任务执行] [成功] 流水线顺利完成！视频已落盘。")
        if not update_task_status(task_data, "success", relative_video_path):
            print("[任务执行] [错误] 视频生成成功，但最终成功状态写入失败。")
    else:
        print("[任务执行] [失败] 流水线执行失败，中断操作。")
        if not update_task_status(task_data, "fail"):
            print("[任务执行] [错误] 最终失败状态写入异常。")
            
    return True

# ==========================================
# 主函数入口 (Main Entrypoint)
# ==========================================

def main() -> None:
    """Colab 服务端守护进程，包含文件系统缓存穿透与死循环监听"""
    print("==========================================")
    print("      基于文件通信的 Colab 服务端已启动      ")
    print("==========================================")
    
    if not mount_google_drive_and_init():
        print("[守护进程] [严重错误] 初始化挂载失败，脚本停止。")
        return
        
    print("\n[守护进程] 系统就绪，开始挂机轮询 task_file.txt...\n")
    
    try:
        while True:
            process_pending_task()
            time.sleep(POLL_INTERVAL_SECONDS)
    except KeyboardInterrupt:
        print("\n[守护进程] 收到手动终止信号 (Ctrl+C)，退出轮询。")

if __name__ == "__main__":
    main()